# Baseline Analysis — PI Direct, PI Indirect, IOH

Loads evidence from `evidence/baseline/*.json`, computes Attack Success Rate (ASR) per
`(variant, category, temperature)` with Wilson 95% CI, and generates heatmaps.

**No live API calls.** All data comes from persisted JSON files.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

EVIDENCE_DIR = Path('../evidence/baseline')
FIGURES_DIR = EVIDENCE_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Evidence directory: {EVIDENCE_DIR.resolve()}')

In [ ]:
records = []
for f in sorted(EVIDENCE_DIR.glob('*.json')):
    if f.name.startswith('_'):
        continue
    try:
        records.append(json.loads(f.read_text(encoding='utf-8')))
    except Exception as e:
        print(f'[WARN] {f.name}: {e}')
df = pd.DataFrame(records)
print(f'Loaded {len(df)} evidence records.')
df.head(3)

In [ ]:
df_ok = df[df['execution_status'] == 'success'].copy()
df_err = df[df['execution_status'] != 'success'].copy()
print(f'Successful executions: {len(df_ok)}')
print(f'Errors / max_iterations: {len(df_err)}')
if len(df_err):
    print(df_err.groupby(['variant', 'category', 'execution_status']).size())

In [ ]:
coverage = (df_ok.groupby(['variant', 'category', 'temperature'])
    .agg(n_success=('success_flag', 'count')).reset_index())
print('Coverage per stratum:')
print(coverage.to_string(index=False))
below_min = coverage[coverage['n_success'] < 45]
if not below_min.empty:
    print('[WARN] Strata below 45 evidences:')
    print(below_min)

In [ ]:
def wilson_ci(n_s, n_t, z=1.96):
    if n_t == 0: return 0.0, 0.0
    p = n_s / n_t
    d = 1 + z**2/n_t
    c = (p + z**2/(2*n_t)) / d
    m = z * np.sqrt(p*(1-p)/n_t + z**2/(4*n_t**2)) / d
    return max(0.0, c-m), min(1.0, c+m)

agg = (df_ok.groupby(['variant','category','temperature'])
    .agg(n_attack=('success_flag','sum'), n_total=('success_flag','count')).reset_index())
agg['asr'] = agg['n_attack'] / agg['n_total']
ci = agg.apply(lambda r: wilson_ci(int(r['n_attack']), int(r['n_total'])), axis=1)
agg['ci_lower'] = ci.apply(lambda x: x[0])
agg['ci_upper'] = ci.apply(lambda x: x[1])
print(agg.to_string(index=False))

In [ ]:
summary_path = EVIDENCE_DIR / 'summary.csv'
agg[['variant','category','temperature','asr','ci_lower','ci_upper','n_attack','n_total']].to_csv(summary_path, index=False)
print(f'Summary CSV: {summary_path}')

In [ ]:
CAT = {'pi_direct':'PI Direta','pi_indirect':'PI Indireta','ioh':'Insecure Output'}
VAR = {'a':'A: Claude Sonnet','b':'B: Llama 8B','c':'C: Guard+Llama+Presidio'}

for temp in [0.0, 0.7]:
    if temp not in agg['temperature'].values: continue
    subset = agg[agg['temperature']==temp]
    pivot = subset.pivot(index='variant', columns='category', values='asr')
    pivot = pivot.rename(index=VAR, columns=CAT)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.0%', vmin=0, vmax=1, cmap='RdYlGn_r', linewidths=0.5)
    ax.set_title(f'ASR Baseline (temp={temp})')
    plt.tight_layout()
    out = FIGURES_DIR / f'heatmap_temp_{temp}.png'
    fig.savefig(out, dpi=150)
    print(f'Heatmap: {out}')
    plt.show()

In [ ]:
# Stratified manual-review sample (10%, min 5 per stratum)
frames = []
for (_v, _c, _t), g in df_ok.groupby(['variant','category','temperature']):
    n_sample = max(5, int(len(g)*0.10))
    frames.append(g.sample(n=min(n_sample, len(g)), random_state=42))
review_df = pd.concat(frames).reset_index(drop=True)
review_df['manual_review'] = pd.NA
review_path = EVIDENCE_DIR / 'manual_review_sample.csv'
review_df[['id','variant','category','temperature','technique','payload','response','success_flag','success_reason','manual_review']].to_csv(review_path, index=False)
print(f'Manual review sample: {review_path} ({len(review_df)} records)')

In [ ]:
# Cohen kappa per category (run after filling manual_review in the CSV)
try:
    from sklearn.metrics import cohen_kappa_score
    reviewed = pd.read_csv(EVIDENCE_DIR / 'manual_review_sample.csv').dropna(subset=['manual_review'])
    if len(reviewed):
        for cat, grp in reviewed.groupby('category'):
            if len(grp) >= 2:
                k = cohen_kappa_score(grp['success_flag'].astype(int), grp['manual_review'].astype(int))
                status = 'OK' if k >= 0.6 else 'BELOW THRESHOLD -- revise heuristic'
                print(f'{cat}: kappa={k:.3f}  {status}')
    else:
        print('Fill manual_review column first.')
except ImportError:
    print('Install scikit-learn: pip install scikit-learn')